# Joining Multiple Seasons: WTA Matches 2010–2023

We have 14 years of match data spread across separate CSV files. This notebook consolidates
them into a single player-season dataset — the input for the regression model in the next notebook.

Data: [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) · CC BY-NC-SA 4.0

In [ ]:
import pandas as pd
import glob

## [ASIDE] Why one file per year?

This is a common pattern in data engineering: partitioning data by time period keeps individual
files manageable and makes incremental updates straightforward (add a new year, don't rewrite
everything). The `glob` module lets us discover files by pattern so we're not hardcoding filenames.

## Exercise 1: Find the Match Files

**Which glob pattern finds exactly the WTA match files (excluding qualifying/ITF files)?**

- A) `"data/*.csv"` — matches all CSVs in data/, including non-match files
- B) `"data/wta_matches_*.csv"` — matches match files AND qualifying/ITF files (e.g. wta_matches_qual_itf_2023.csv)
- C) `"data/wta_matches_[0-9]*.csv"` — matches only files where the character after the last underscore is a digit (year files only) ✓
- D) `"data/**/*.csv"` — recurses into subdirectories, overkill here

Set `answer` below.

In [ ]:
answer = "___"
assert answer == "C", "B is close but also picks up qual/ITF files. C restricts to year-numbered files."
print("Correct. The [0-9] character class ensures we only match year files.")

In [ ]:
all_files = glob.___("data/wta_matches_[0-9]*.csv")  # fill in the glob function name
# Filter to 2010–2023
files = [f for f in all_files if any(str(yr) in f for yr in range(2010, 2024))]
files = sorted(files)
print(f"Found {len(files)} files")
print(files[:3], "...")

## [ASIDE] Loops vs copy-paste

If you've ever applied the same transformation across 14 Excel sheets manually, a loop is Python's
answer. Same logic, none of the repetition. The list we're building (`dfs`) collects all the
DataFrames before we concatenate — more efficient than concatenating inside the loop.

## Exercise 2: Load and Concatenate

Loop over the files, load each one, collect into a list, then concatenate.

In [ ]:
dfs = []
# Step 1: write a for-loop over `files`

    # Step 2: inside the loop — read each CSV with pd.read_csv(filepath, low_memory=False)
    # Step 3: append the resulting DataFrame to dfs

# Step 4: concatenate dfs into a single DataFrame
#         use pd.concat with ignore_index=True
df = pd.___(dfs, ignore_index=___)
print(f"Combined shape: {df.shape}")

## [ASIDE] `ignore_index=True`

Without it, `pd.concat` preserves each file's original row index — giving you duplicate index
values (multiple rows all labelled 0, 1, 2...). `ignore_index=True` resets to a clean
sequential index across the combined DataFrame. Usually what you want after a concat.

## Exercise 3: Inspect the Combined Data

In [ ]:
print(f"Shape: {df.shape}")
missing = df.isnull().sum().sort_values(ascending=False)
print(f"\nTop 10 columns by missingness:\n{missing.head(10)}")

## Exercise 4: Filter and Select

Keep only rows with non-null values in the key stat columns.
Then select the columns we need for feature engineering.

In [ ]:
STAT_COLS = ["w_ace", "w_svpt", "w_1stIn", "w_df", "l_ace", "l_svpt", "l_1stIn", "l_df"]
# Step 1: drop rows where any column in STAT_COLS is null
# hint: df.dropna(subset=STAT_COLS)

KEEP_COLS = [
    "tourney_date", "surface",
    "winner_name", "winner_rank", "winner_rank_points",
    "w_ace", "w_svpt", "w_1stIn", "w_df",
    "loser_name", "loser_rank", "loser_rank_points",
    "l_ace", "l_svpt", "l_1stIn", "l_df",
]
# Step 2: select only KEEP_COLS from df, assign back to df
# hint: df = df[KEEP_COLS].copy()
print(f"Shape after filtering: {df.shape}")

## [ASIDE] Feature selection early

Carrying unnecessary columns through an analysis slows things down and makes DataFrames
harder to read. Selecting only what you need at the point of loading is a habit worth
building — the equivalent of writing a targeted SQL SELECT rather than SELECT *.

## Exercise 5: Build a Player-Centric View

The match data has one row per match with winner-prefixed and loser-prefixed columns.
To aggregate by player, we need to reshape: create one row per player appearance
(win or loss) with consistent column names.

In [ ]:
def build_player_matches(df: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    df : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        ...
    """
    # Your code here
    pass


player_matches = build_player_matches(df)
print(f"Player-match rows: {player_matches.shape}")
player_matches.head()

## Exercise 6: Compute Per-Match Rates

Add derived rate columns before aggregating.

In [ ]:
player_matches["first_serve_pct"] = player_matches["___"] / player_matches["___"]
player_matches["ace_rate"] = player_matches["___"] / player_matches["svpt"]
player_matches["df_rate"] = player_matches["___"] / player_matches["svpt"]

## Exercise 7: Aggregate to Player-Season Level

In [ ]:
def aggregate_player_season(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    player_matches : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        Columns: player, year, win_count, total_matches, win_rate,
                 avg_first_serve_pct, avg_ace_rate, avg_df_rate, avg_rank.
    """
    # Your code here
    pass


season_stats = aggregate_player_season(player_matches)
print(f"Player-season rows: {season_stats.shape}")
season_stats.head()

## Exercise 8: Season-End Ranking Points

For each player-season, take the ranking points from their last match of the year.
This is a proxy for year-end ranking — not official, but a good approximation.

In [ ]:
def get_season_end_points(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    [Write your docstring here]

    Parameters
    ----------
    player_matches : pd.DataFrame
        ...

    Returns
    -------
    pd.DataFrame
        Columns: player, year, ranking_points.
    """
    # Your code here
    pass


season_end_points = get_season_end_points(player_matches)
season_end_points.head()

## Exercise 9: Merge and Write Output

In [ ]:
compiled = season_stats.___(season_end_points, on=["player", "year"], how="inner")
compiled = compiled.dropna()
print(f"Final compiled shape: {compiled.shape}")
compiled.head()

In [ ]:
OUTPUT_PATH = "output/wta_compiled.csv"  # output/ relative to project root
compiled.to_csv(___, index=___)
print(f"Written to {OUTPUT_PATH}")

## Git Signpost

Commit the notebook. The compiled CSV lives in `output/` which is gitignored — the learner
regenerates it by running this notebook.

```bash
git add JOIN_DATA.ipynb
git commit -m "complete JOIN_DATA notebook"
```